# Work IQ actions

The [Work IQ MCP notebook](workiq-mcp.ipynb) covers read tools. This optional notebook focuses on the four tools that change Microsoft 365 data: `create_entity`, `update_entity`, `delete_entity`, and `do_action`.

**Writes affect real data as the signed-in user.** Each action previews its target and payload, then asks you to type `yes` before executing. Any other response skips that action. The practice event has no attendees, and its ID is reused for update and cleanup. Sending mail requires a recipient and a separate confirmation.

> In the Skillable lab, Work IQ authentication is already configured. For self-deployment, complete the [Work IQ admin setup](../docs/workiq-admin-setup.md) first.

## Step 1: Authenticate as the signed-in user

Load the app registration from `.env`. Sign in with your lab account in the browser when prompted.

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from msal import PublicClientApplication

load_dotenv(override=True)

app_id = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_SEARCH_ENTRA_TENANT_ID"]

app = PublicClientApplication(
    client_id=app_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
auth_result = app.acquire_token_interactive(["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
print("Signed in to Work IQ")


async def call_workiq(tool_name, arguments):
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token}"},
        timeout=httpx.Timeout(30, read=180),
    ) as http_client:
        async with streamable_http_client(
            "https://workiq.svc.cloud.microsoft/mcp", http_client=http_client
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
    if result.isError:
        raise RuntimeError(f"Work IQ {tool_name} failed: {result.content}")
    return result


def event_from_result(result):
    payloads = [result.structuredContent]
    for content in result.content:
        if content.type == "text":
            try:
                payloads.append(json.loads(content.text))
            except json.JSONDecodeError:
                continue
    for payload in payloads:
        if isinstance(payload, dict):
            for candidate in [payload, *(payload.get(key) for key in ("body", "data", "value"))]:
                if isinstance(candidate, dict) and isinstance(candidate.get("id"), str) and candidate["id"].strip():
                    return candidate
    return None

## Step 2: Inspect write tools and schemas

List the write tools and their input schemas, then inspect the create schema for calendar events. The collection path `/me/events` supports create; update targets an individual event instead. These discovery calls do not change data.

- `create_entity` takes a collection's `parentUrl`.
- `update_entity` and `delete_entity` take an item's `entityUrl`.
- `do_action` takes an `actionUrl`, such as `/me/sendMail`.
- `jsonBody` is a JSON-encoded string, so encode payloads with `json.dumps`.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()
            for tool in tools_response.tools:
                if tool.name in {"create_entity", "update_entity", "delete_entity", "do_action"}:
                    print(tool.name, json.dumps(tool.inputSchema, indent=2))

            for operation in ("create",):
                result = await session.call_tool("get_schema", {"path": "/me/events", "operationType": operation})
                if result.isError:
                    raise RuntimeError(f"Work IQ get_schema failed: {result.content}")
                print(f"Calendar event {operation} schema:")
                for content in result.content:
                    if content.type == "text":
                        print(content.text)

## Step 3: Create a practice calendar event

Review the proposed event for tomorrow with no attendees, then type `yes` to create it. The response supplies the event ID for the next cells automatically. If the response format is not recognized, copy only the created event's ID when prompted. Open your calendar to inspect the event before continuing.

In [ ]:
from datetime import datetime, timedelta, timezone

if globals().get("created_event_result") is not None:
    raise ValueError("A practice event was already created. Continue to update or cleanup instead of creating a duplicate.")

start = (datetime.now(timezone.utc) + timedelta(days=1)).replace(hour=15, minute=0, second=0, microsecond=0)
end = start + timedelta(minutes=30)
create_args = {
    "parentUrl": "/me/events",
    "jsonBody": json.dumps({
        "subject": "Work IQ lab practice",
        "start": {"dateTime": start.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
        "end": {"dateTime": end.strftime("%Y-%m-%dT%H:%M:%S"), "timeZone": "UTC"},
    }),
}
print(json.dumps(create_args, indent=2))

if input("Create this event in your calendar? Type yes to confirm: ").strip().lower() == "yes":
    created_event_result = await call_workiq("create_entity", create_args)
    created_event = event_from_result(created_event_result)
    event_id = created_event["id"] if created_event else ""
    if not event_id:
        print("Event created, but its ID could not be extracted. Do not create it again.")
        print(created_event_result.model_dump_json(indent=2))
        event_id = input("Paste the created practice event's ID from the response: ").strip()
    print(f"Created: Work IQ lab practice | {start:%Y-%m-%d %H:%M} UTC | 30 minutes | no attendees")
    print(f"Event ID: {event_id or 'not captured; set event_id before update or cleanup'}")
    if created_event and created_event.get("webLink"):
        print(f"Open event: {created_event['webLink']}")
else:
    print("Skipped: no event created.")

## Step 4: Update the practice event

This cell uses the ID captured in step 3. Review the target and proposed subject change, then type `yes` to update the practice event. Refresh your calendar to inspect the changed subject before cleanup.

In [ ]:
if globals().get("created_event_result") is None or not globals().get("event_id", "").strip():
    raise ValueError("Create the practice event in step 3 and capture its event_id before updating it.")

update_args = {
    "entityUrl": f"/me/events/{event_id}",
    "jsonBody": json.dumps({"subject": "Work IQ lab practice (updated)"}),
}
print(json.dumps(update_args, indent=2))

if input("Update this practice event? Type yes to confirm: ").strip().lower() == "yes":
    update_result = await call_workiq("update_entity", update_args)
    print(f"Updated event {event_id}: Work IQ lab practice (updated)")
else:
    print("Skipped: event unchanged.")

## Step 5: Clean up the practice event

Review the captured practice event ID, then type `yes` to delete only that event. Successful cleanup clears the ID so rerunning update or delete cannot accidentally reuse it.

In [ ]:
if globals().get("created_event_result") is None or not globals().get("event_id", "").strip():
    raise ValueError("There is no captured practice event to delete. Create one in step 3 first.")

delete_args = {"entityUrl": f"/me/events/{event_id}"}
print(json.dumps(delete_args, indent=2))

if input("Delete this practice event? Type yes to confirm: ").strip().lower() == "yes":
    delete_result = await call_workiq("delete_entity", delete_args)
    print(f"Deleted practice event: {event_id}")
    event_id = ""
    created_event_result = None
else:
    print("Skipped: practice event remains in your calendar.")

## Step 6: Send a practice email

`do_action` invokes an action such as sending mail, replying, or accepting an event. Inspect its tool schema from step 2 and the [Microsoft Graph sendMail request](https://learn.microsoft.com/graph/api/user-sendmail) before approving the payload.

Enter your own lab email address as the recipient. Review the address and message, then type `yes` to send it. Leave the address blank to skip. Check your inbox and Sent Items after sending.

In [ ]:
recipient = input("Your own lab email address (blank to skip): ").strip()
send_args = {
    "actionUrl": "/me/sendMail",
    "jsonBody": json.dumps({
        "message": {
            "subject": "Work IQ lab practice",
            "body": {"contentType": "Text", "content": "This is a practice message from the Work IQ lab."},
            "toRecipients": [{"emailAddress": {"address": recipient}}],
        },
        "saveToSentItems": True,
    }),
}
print(json.dumps(send_args, indent=2))

if recipient and input(f"Send this email to {recipient}? Type yes to confirm: ").strip().lower() == "yes":
    send_result = await call_workiq("do_action", send_args)
    print(f"Send request accepted for {recipient}: Work IQ lab practice. Check your inbox and Sent Items.")
else:
    print("Skipped: no mail sent.")

## Recap

The write workflow is **inspect schema -> draft -> confirm -> write**. Each confirmed action performs a real operation: create a practice event, update its subject, delete it, and send a practice email. Declined actions do not execute. If you skipped cleanup, return to step 5 to remove your practice event.

**Optional next step:** [Work IQ A2A](workiq-a2a.ipynb) demonstrates delegating a task to Work IQ as a remote agent.